# 01 Data Acquisition

Notebook นี้แสดงกระบวนการเก็บข้อมูลทั้งหมดโดยตรง ตั้งแต่การอ่าน MasterData การส่ง request รายเดือน การดาวน์โหลดข้อมูลภายนอก การตรวจความครบถ้วน ไปจนถึงการบันทึกแหล่งข้อมูลและ checksum

แหล่งข้อมูลประกอบด้วย

- ดัชนีราคาสินค้านำเข้า (IMI) จากสำนักงานนโยบายและยุทธศาสตร์การค้า
- อัตราแลกเปลี่ยนบาทต่อดอลลาร์ `DEXTHUS` จาก FRED
- ราคาน้ำมัน Brent `DCOILBRENTEU` จาก FRED

ข้อมูลใน `data/raw/` ไม่มีการแก้ไขหรือเขียนทับ หากต้องการดาวน์โหลดใหม่ให้เปลี่ยน `DOWNLOAD_NEW` เป็น `True` ระบบจะสร้างชื่อไฟล์ใหม่ด้วย timestamp UTC

## 1. การตั้งค่าและฟังก์ชันพื้นฐาน

ค่าเริ่มต้นใช้ snapshot ที่ดาวน์โหลดและตรวจสอบแล้ว เพื่อป้องกันการยิง API ซ้ำเมื่อกด Run All ระหว่างตรวจงาน หากต้องการดึงข้อมูลใหม่จริงให้กำหนด `DOWNLOAD_NEW = True`

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DOWNLOAD_NEW = False
REQUEST_DELAY_SECONDS = 0.30

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'PROJECT_SCOPE.md').exists()
)
RAW_DIR = ROOT / 'data' / 'raw'
SOURCES_PATH = RAW_DIR / 'sources.json'
MASTER_URL = 'https://index-api.tpso.go.th/OpenApi/Imex/Month/MasterData'
MONTH_URL = 'https://index-api.tpso.go.th/OpenApi/Imex/Month'
FRED_SERIES = {
    'DEXTHUS': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXTHUS',
    'DCOILBRENTEU': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU',
}

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def make_session() -> requests.Session:
    retry = Retry(
        total=5, connect=5, read=5, backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET', 'POST'}),
        respect_retry_after_header=True,
    )
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers['User-Agent'] = 'imi-early-warning/1.0 (academic project)'
    return session

def enumerate_months(start_year, start_month, end_year, end_month):
    result = []
    year, month = start_year, start_month
    while (year, month) <= (end_year, end_month):
        result.append((year, month))
        month += 1
        if month == 13:
            year, month = year + 1, 1
    return result

session = make_session()
downloaded_at = datetime.now(timezone.utc)
stamp = downloaded_at.strftime('%Y%m%dT%H%M%SZ')
print(f'Project root: {ROOT}')
print(f'Download new data: {DOWNLOAD_NEW}')

Project root: /Users/pattarapol/itkmitl/imi-early-warning
Download new data: False


## 2. อ่าน MasterData ของ IMI

MasterData เป็นตัวกำหนดปีฐาน เดือนเริ่มต้น เดือนล่าสุด และรายชื่อรหัสสินค้า จึงไม่กำหนดข้อมูลเหล่านี้แบบ hard-code

In [2]:
sources_document = json.loads(SOURCES_PATH.read_text(encoding='utf-8'))

if DOWNLOAD_NEW:
    response = session.get(MASTER_URL, timeout=60)
    response.raise_for_status()
    master_data = response.json()
    master_path = RAW_DIR / f'imi_master_{stamp}.json'
    master_path.write_text(
        json.dumps(master_data, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
else:
    imi_source = next(
        item for item in reversed(sources_document['sources'])
        if item['source_id'].startswith('imi-monthly-')
    )
    stamp = imi_source['source_id'].removeprefix('imi-monthly-')
    master_source = next(
        item for item in sources_document['sources']
        if item['source_id'] == f'imi-master-{stamp}'
    )
    master_path = RAW_DIR / master_source['file']
    master_data = json.loads(master_path.read_text(encoding='utf-8'))

assert isinstance(master_data, list)
imi_master = next(item for item in master_data if item['type'] == 'IMI')
periods = imi_master['dataAvailablePeriods']
assert len(periods) == 1 and periods[0]['periodType'] == 'month'
period = periods[0]

year_base = int(period['yearBase'])
start_year_be = int(period['startYear'])
start_month = int(period['startPeriod'])
end_year_be = int(period['endYear'])
end_month = int(period['endPeriod'])
commodity_codes = [str(item['code']).zfill(3) for item in imi_master['commodities']]
months = enumerate_months(start_year_be, start_month, end_year_be, end_month)

master_summary = pd.DataFrame([{
    'ปีฐาน (พ.ศ.)': year_base,
    'เดือนเริ่มต้น': f'{start_year_be - 543:04d}-{start_month:02d}',
    'เดือนล่าสุด': f'{end_year_be - 543:04d}-{end_month:02d}',
    'จำนวนเดือน': len(months),
    'จำนวนรหัส IMI': len(commodity_codes),
}])
display(master_summary)

,ปีฐาน (พ.ศ.),เดือนเริ่มต้น,เดือนล่าสุด,จำนวนเดือน,จำนวนรหัส IMI
0,2555,2000-01,2026-08,320,81


## 3. ดาวน์โหลด IMI รายเดือน

API รับหนึ่งเดือนต่อ request และสามารถส่งรหัสสินค้าทั้งหมดพร้อมกันได้ แต่ละ request จึงประกอบด้วยปีฐาน ปี พ.ศ. เดือน ประเภท `IMI` และรหัสสินค้าทั้ง 81 รหัส มีการหน่วงเวลา 0.30 วินาทีและ retry สำหรับ HTTP 429 และ server error

In [3]:
if DOWNLOAD_NEW:
    imi_rows = []
    request_manifest = []
    for sequence, (year_be, month) in enumerate(months, start=1):
        body = {
            'yearBase': year_base,
            'year': year_be,
            'month': month,
            'type': 'IMI',
            'commodities': commodity_codes,
        }
        response = session.post(MONTH_URL, json=body, timeout=60)
        response.raise_for_status()
        payload = response.json()
        if not isinstance(payload, list) or not payload:
            raise ValueError(f'ไม่มีข้อมูล พ.ศ. {year_be}-{month:02d}')
        if any(int(row['year']) != year_be or int(row['month']) != month for row in payload):
            raise ValueError(f'เดือนในผลตอบกลับไม่ตรงกับ request: {year_be}-{month:02d}')
        imi_rows.extend(payload)
        request_manifest.append({
            'sequence': sequence, 'body': body, 'row_count': len(payload)
        })
        if sequence % 40 == 0 or sequence == len(months):
            print(f'{sequence}/{len(months)} เดือน; สะสม {len(imi_rows):,} แถว')
        if sequence < len(months):
            time.sleep(REQUEST_DELAY_SECONDS)

    manifest_path = RAW_DIR / f'imi_request_manifest_{stamp}.json'
    manifest_path.write_text(
        json.dumps(request_manifest, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    imi_df = pd.DataFrame(imi_rows)
    imi_path = RAW_DIR / f'imi_monthly_{stamp}.csv'
    imi_df.to_csv(imi_path, index=False, encoding='utf-8')
else:
    imi_path = RAW_DIR / imi_source['file']
    manifest_path = RAW_DIR / imi_source['request_bodies_file']
    imi_df = pd.read_csv(imi_path, dtype={'commodityCode': 'string'})
    imi_df['commodityCode'] = imi_df['commodityCode'].str.zfill(3)
    request_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

required_columns = {
    'commodityCode', 'commodityNameTH', 'month', 'year', 'yearBase',
    'index', 'change', 'changeYear', 'changeAVG', 'level', 'createdAt',
}
assert required_columns <= set(imi_df.columns)
assert not imi_df.duplicated(['commodityCode', 'year', 'month']).any()
returned_months = set(zip(imi_df['year'].astype(int), imi_df['month'].astype(int)))
assert returned_months == set(months)
assert len(request_manifest) == len(months)

group_coverage = (
    imi_df[imi_df['commodityCode'].isin(['000', '100', '318', '402'])]
    .groupby('commodityCode')
    .agg(rows=('index', 'size'), missing_index=('index', lambda x: x.isna().sum()))
    .reset_index()
)
assert (group_coverage['rows'] == len(months)).all()
display(group_coverage)
print(f'IMI ทั้งหมด {len(imi_df):,} แถว จาก {imi_df["commodityCode"].nunique()} รหัส')

,commodityCode,rows,missing_index
0,000,320,0
1,100,320,0
2,318,320,0
3,402,320,0


IMI ทั้งหมด 25,752 แถว จาก 81 รหัส


## 4. ดาวน์โหลดข้อมูลรายวันจาก FRED

ใช้ไฟล์ CSV ที่ไม่ต้องใช้ API key และเก็บ byte stream จากต้นทางโดยตรง ระหว่างการรันจริงพบว่า Python `requests` เกิด read timeout กับ FRED ขณะที่ `curl` ดาวน์โหลด URL เดียวกันสำเร็จ จึงใช้ `curl` พร้อม timeout และ retry สำหรับสองชุดข้อมูลนี้

In [4]:
fred_frames = {}
fred_paths = {}
target_end = pd.Timestamp(year=end_year_be - 543, month=end_month, day=1)

for series_id, url in FRED_SERIES.items():
    if DOWNLOAD_NEW:
        completed = subprocess.run(
            [
                'curl', '--fail', '--silent', '--show-error', '--location',
                '--connect-timeout', '15', '--max-time', '60',
                '--retry', '3', '--retry-all-errors', url,
            ],
            check=True, capture_output=True, timeout=75,
        )
        if not completed.stdout:
            raise ValueError(f'{series_id}: FRED ส่งไฟล์ว่างกลับมา')
        path = RAW_DIR / f'{series_id.lower()}_daily_{stamp}.csv'
        path.write_bytes(completed.stdout)
        frame = pd.read_csv(io.BytesIO(completed.stdout), na_values=['.'])
    else:
        source = next(
            item for item in sources_document['sources']
            if item['source_id'] == f'{series_id.lower()}-{stamp}'
        )
        path = RAW_DIR / source['file']
        frame = pd.read_csv(path, na_values=['.'])

    assert list(frame.columns) == ['observation_date', series_id]
    frame['observation_date'] = pd.to_datetime(frame['observation_date'], errors='raise')
    assert frame['observation_date'].is_monotonic_increasing
    assert frame['observation_date'].min() <= pd.Timestamp('2000-01-01')
    assert frame['observation_date'].max() >= target_end
    fred_frames[series_id] = frame
    fred_paths[series_id] = path

fred_summary = pd.DataFrame([
    {
        'series': series_id,
        'rows': len(frame),
        'non_null': int(frame[series_id].notna().sum()),
        'missing': int(frame[series_id].isna().sum()),
        'start': frame['observation_date'].min().date(),
        'end': frame['observation_date'].max().date(),
    }
    for series_id, frame in fred_frames.items()
])
display(fred_summary)

,series,rows,non_null,missing,start,end
0,DEXTHUS,11931,11395,536,1981-01-02,2026-09-25
1,DCOILBRENTEU,10270,9097,1173,1987-05-20,2026-09-29


## 5. บันทึก metadata และ checksum

เมื่อดาวน์โหลดชุดใหม่ จะเพิ่มรายการใน `sources.json` หลังจากข้อมูลทั้งหมดผ่าน validation แล้วเท่านั้น แต่ละรายการระบุ URL, method, request body หรือ manifest, เวลาดาวน์โหลด, จำนวนแถว, ช่วงข้อมูล, ใบอนุญาต และ SHA-256

In [5]:
if DOWNLOAD_NEW:
    downloaded_at_iso = downloaded_at.isoformat().replace('+00:00', 'Z')
    new_records = [
        {
            'source_id': f'imi-master-{stamp}',
            'name': 'IMI MasterData',
            'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'url': MASTER_URL, 'method': 'GET',
            'downloaded_at_utc': downloaded_at_iso,
            'row_count': len(master_data), 'data_range': None,
            'file': master_path.name, 'sha256': file_sha256(master_path),
            'license': 'Open Data Common',
            'citation': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. ข้อมูลดัชนีราคาสินค้านำเข้าของประเทศ. ศูนย์กลางข้อมูลเปิดภาครัฐ.',
        },
        {
            'source_id': f'imi-monthly-{stamp}',
            'name': 'ดัชนีราคาสินค้านำเข้ารายเดือน (IMI)',
            'owner': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'publisher': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์',
            'url': MONTH_URL, 'method': 'POST',
            'request_body_template': {
                'yearBase': year_base, 'year': '<พ.ศ.>', 'month': '<1-12>',
                'type': 'IMI', 'commodities': commodity_codes,
            },
            'request_bodies_file': manifest_path.name,
            'request_bodies_sha256': file_sha256(manifest_path),
            'request_count': len(request_manifest), 'failed_request_count': 0,
            'downloaded_at_utc': downloaded_at_iso, 'row_count': len(imi_df),
            'data_range': {
                'start': f'{start_year_be - 543:04d}-{start_month:02d}',
                'end': f'{end_year_be - 543:04d}-{end_month:02d}',
                'calendar_in_file': 'Buddhist Era',
            },
            'file': imi_path.name, 'sha256': file_sha256(imi_path),
            'license': 'Open Data Common',
            'citation': 'สำนักงานนโยบายและยุทธศาสตร์การค้า กระทรวงพาณิชย์. ข้อมูลดัชนีราคาสินค้านำเข้าของประเทศ. ศูนย์กลางข้อมูลเปิดภาครัฐ.',
        },
    ]
    fred_metadata = {
        'DEXTHUS': ('Thai Baht to U.S. Dollar Spot Exchange Rate', 'Board of Governors of the Federal Reserve System (US)'),
        'DCOILBRENTEU': ('Crude Oil Prices: Brent - Europe', 'U.S. Energy Information Administration'),
    }
    for series_id, frame in fred_frames.items():
        name, owner = fred_metadata[series_id]
        new_records.append({
            'source_id': f'{series_id.lower()}-{stamp}', 'name': name,
            'owner': owner, 'publisher': 'FRED, Federal Reserve Bank of St. Louis',
            'url': FRED_SERIES[series_id], 'method': 'GET',
            'downloaded_at_utc': downloaded_at_iso, 'row_count': len(frame),
            'non_null_row_count': int(frame[series_id].notna().sum()),
            'data_range': {
                'start': str(frame['observation_date'].min().date()),
                'end': str(frame['observation_date'].max().date()),
            },
            'file': fred_paths[series_id].name,
            'sha256': file_sha256(fred_paths[series_id]),
            'license': 'Public Domain: Citation Requested',
            'citation': f'{owner}, {name} [{series_id}], retrieved from FRED, Federal Reserve Bank of St. Louis.',
        })
    sources_document['sources'].extend(new_records)
    sources_document['last_updated_at_utc'] = downloaded_at_iso
    SOURCES_PATH.write_text(
        json.dumps(sources_document, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
else:
    new_records = [
        item for item in sources_document['sources']
        if item['source_id'].endswith(stamp)
    ]

metadata_summary = pd.DataFrame([
    {
        'source_id': item['source_id'],
        'file': item['file'],
        'rows': item['row_count'],
        'sha256': item['sha256'][:12] + '...',
    }
    for item in new_records
])
display(metadata_summary)

,source_id,file,rows,sha256
0,imi-master-20261005T054207Z,imi_master_20261005T054207Z.json,2,618ad4c6de8d...
1,imi-monthly-20261005T054207Z,imi_monthly_20261005T054207Z.csv,25752,55a338b42c21...
2,dexthus-20261005T054207Z,dexthus_daily_20261005T054207Z.csv,11931,82b99fc8c2e1...
3,dcoilbrenteu-20261005T054207Z,dcoilbrenteu_daily_20261005T054207Z.csv,10270,68b1b12a3e21...


## 6. Checkpoint ก่อนเข้าสู่ Data Understanding

เงื่อนไขผ่านคือ IMI ครบทุกเดือนตาม MasterData ไม่มี request ล้มเหลว กลุ่มหลักทั้งสี่รหัสมีข้อมูลครบ ข้อมูล FRED ครอบคลุมตั้งแต่ปี 2000 ถึงเดือนล่าสุดของ IMI และ checksum ตรงกับ `sources.json`

In [6]:
record_by_file = {item['file']: item for item in new_records}
files_to_verify = [master_path, imi_path, *fred_paths.values()]
checksum_results = []
for path in files_to_verify:
    expected = record_by_file[path.name]['sha256']
    actual = file_sha256(path)
    checksum_results.append({
        'file': path.name, 'checksum_match': actual == expected
    })
checksum_df = pd.DataFrame(checksum_results)
display(checksum_df)

assert checksum_df['checksum_match'].all()
assert len(months) == len(request_manifest)
assert (group_coverage['rows'] == len(months)).all()
assert all(frame['observation_date'].max() >= target_end for frame in fred_frames.values())

checkpoint = pd.DataFrame([{
    'IMI rows': len(imi_df),
    'IMI codes': imi_df['commodityCode'].nunique(),
    'IMI months': len(months),
    'Failed requests': 0,
    'DEXTHUS rows': len(fred_frames['DEXTHUS']),
    'Brent rows': len(fred_frames['DCOILBRENTEU']),
    'Checksum files passed': int(checksum_df['checksum_match'].sum()),
}])
display(checkpoint)
print('CHECKPOINT PASSED: ข้อมูลทั้งสามแหล่งครบตามเงื่อนไขของโครงการ')

if DOWNLOAD_NEW:
    fx = fred_frames['DEXTHUS']
    brent = fred_frames['DCOILBRENTEU']
    log_entry = f'''
## [{downloaded_at.date().isoformat()}] ขั้นตอน: 01 Data Acquisition
**สิ่งที่ทำ:** ดาวน์โหลด IMI ทุกรหัสตาม MasterData จำนวน {len(months):,} เดือน ดาวน์โหลด DEXTHUS และ DCOILBRENTEU จาก FRED ตรวจ schema และบันทึก metadata พร้อม SHA-256

**ผลที่ได้:** IMI จำนวน {len(imi_df):,} แถว ครอบคลุม {start_year_be - 543:04d}-{start_month:02d} ถึง {end_year_be - 543:04d}-{end_month:02d} จาก {len(commodity_codes):,} รหัสและ {len(request_manifest):,} requests โดยไม่มี request ล้มเหลว; DEXTHUS จำนวน {len(fx):,} แถว ช่วง {fx['observation_date'].min().date()} ถึง {fx['observation_date'].max().date()}; DCOILBRENTEU จำนวน {len(brent):,} แถว ช่วง {brent['observation_date'].min().date()} ถึง {brent['observation_date'].max().date()}

**สิ่งที่พบ / ปัญหา:** เก็บค่าว่างของข้อมูลรายวันตามต้นฉบับและไม่แทนด้วยศูนย์

**การตัดสินใจและเหตุผล:** เก็บ raw file ด้วย timestamp UTC และ request manifest แยกต่างหากเพื่อป้องกันการเขียนทับและตรวจสอบย้อนกลับได้

**ขั้นต่อไป:** ดำเนินการ 02 Data Understanding
'''
    with (ROOT / 'logs' / 'process_log.md').open('a', encoding='utf-8') as handle:
        handle.write(log_entry)
    print('บันทึกผลรอบใหม่ลง logs/process_log.md แล้ว')

,file,checksum_match
0,imi_master_20261005T054207Z.json,True
1,imi_monthly_20261005T054207Z.csv,True
2,dexthus_daily_20261005T054207Z.csv,True
3,dcoilbrenteu_daily_20261005T054207Z.csv,True


,IMI rows,IMI codes,IMI months,Failed requests,DEXTHUS rows,Brent rows,Checksum files passed
0,25752,81,320,0,11931,10270,4


CHECKPOINT PASSED: ข้อมูลทั้งสามแหล่งครบตามเงื่อนไขของโครงการ


## 7. Process log

ผลการดาวน์โหลดจริง จำนวนแถว ปัญหา FRED timeout และเหตุผลที่เลือกวิธีแก้ไข ถูกบันทึกไว้ใน `logs/process_log.md` แล้ว การเปิด notebook ในโหมดอ่าน snapshot จะไม่เขียน log ซ้ำ ส่วนโหมดดาวน์โหลดใหม่จะเขียนผลต่อท้ายโดยอัตโนมัติหลังผ่าน checkpoint